# 🍛 Food Vision — Step 2: Leakage Check + Tracked Experiments (MLflow)

**Author:** Ankit Parate · **Repo:** [github.com/ankitparate05/food-vision](https://github.com/ankitparate05/food-vision)

All logic lives in the repo (`src/foodvision/`), embedded here as a snapshot, so Kaggle, CI and the API run the same code.

| Step | What runs | Why |
|------|-----------|-----|
| 1 | Unpack embedded code snapshot | Reproducible code at a known git commit, no GitHub access needed |
| 2 | `scripts/check_duplicates.py` | Find photos that appear in both train and test, which would inflate the score |
| 3 | 3 training runs via `python -m foodvision.train` | Compare models fairly on the **same versioned splits** (`data/splits/`) |
| 4 | MLflow comparison | Pick the winner by **validation** accuracy (test stays untouched for the final report) |
| 5 | Export best model | `best_artifacts/`, ready for the API |

### How to run
1. **Add Input** → dataset *Indian Food Classification* (Pushkar Jain / l33tc0d3r).
2. **Session options** → Accelerator **GPU T4 x2**, Internet **On**.
3. **Save Version → Save & Run All (Commit)**. Takes about **60–75 minutes**. You can close the tab.
4. When done: version page → **Output** → download `best_artifacts/`, `experiments_summary.csv`, `duplicates/`, `mlflow.db`.

## 1. Setup
The repo code (`src/`, `scripts/`, `data/splits/`) is **embedded below as a snapshot** of git commit `588d7c5`,
so this notebook runs without downloading anything from GitHub.

In [ ]:
# Embedded code snapshot (zip, base64): src/foodvision, scripts, data/splits, pyproject.toml
CODE_COMMIT = "588d7c5"
CODE_ZIP_B64 = "..."

In [ ]:
import os, json, shutil, subprocess, sys, base64, io, zipfile, importlib.util
from pathlib import Path

# FV_* variables only exist for local testing; on Kaggle the defaults apply.
REPO_DIR = Path(os.environ.get("FV_REPO_DIR", "/tmp/food-vision"))
DATA_DIR = os.environ.get("FV_DATA_DIR", "/kaggle/input")
WORK     = Path(os.environ.get("FV_WORK", "/kaggle/working"))
QUICK    = os.environ.get("FV_QUICK") == "1"   # tiny runs for testing the notebook itself

if not REPO_DIR.exists():
    with zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_ZIP_B64))) as z:
        z.extractall(REPO_DIR)
if importlib.util.find_spec("mlflow") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mlflow"], check=True)

# Make the package importable for the training subprocesses (no pip install needed)
os.environ["PYTHONPATH"] = str(REPO_DIR / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
sys.path.insert(0, str(REPO_DIR / "src"))
os.chdir(REPO_DIR)
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
COMMIT = CODE_COMMIT
TRACKING_URI = f"sqlite:///{WORK / 'mlflow.db'}"
WORK.mkdir(parents=True, exist_ok=True)
print("Code snapshot:", COMMIT, "| files:", sum(1 for _ in REPO_DIR.rglob('*.py')), "py | outputs ->", WORK)

## 2. Data-leakage check (duplicate images across splits)
Each image gets a 64-bit perceptual hash (dHash). Pairs that differ in ≤ 4 bits are near-duplicates: the same photo resized or re-compressed.
- **train ↔ test pairs** mean the model may have *seen* a test image during training, so the test score is optimistic.
- Leaking val/test copies go into `exclude.csv`, and every run below drops them, **so all models are compared on clean val/test sets**.

In [ ]:
DUP_DIR = WORK / "duplicates"
subprocess.run([sys.executable, "scripts/check_duplicates.py", "--data-dir", DATA_DIR,
                "--splits-dir", "data/splits", "--out", str(DUP_DIR)], check=True)
dup_summary = json.loads((DUP_DIR / "summary.json").read_text())
dup_summary

In [ ]:
import pandas as pd
from IPython.display import Image as ShowImage, display

pairs = pd.read_csv(DUP_DIR / "pairs.csv")
cross = pairs[pairs.split_a != pairs.split_b]
print(f"{len(cross)} cross-split duplicate pairs | label conflicts: {dup_summary['label_conflicts']}")
display(cross.head(15))
if (DUP_DIR / "examples.png").exists():
    display(ShowImage(filename=str(DUP_DIR / "examples.png"), width=520))

## 3. Experiments
All runs share the same splits, exclusions, epochs and seed, so differences come from the **one thing changed per run**.

| Run | Model | Augmentation | Question it answers |
|-----|-------|--------------|---------------------|
| A | EfficientNet-B0 | basic | Baseline on the cleaned splits (reproduces Step 1) |
| B | EfficientNet-B0 | strong (TrivialAugment + RandomErasing) | Does stronger augmentation close the 99% train / 91% val gap? |
| C | ConvNeXt-Tiny | strong | Does a bigger, more modern backbone help? (7x the parameters) |

ConvNeXt uses a lower fine-tune LR (1e-4); large models are easier to damage with big updates.

In [ ]:
RUNS = [
    {"name": "A_effnet_basic",    "model": "efficientnet_b0", "aug": "basic",  "finetune_lr": 3e-4},
    {"name": "B_effnet_strong",   "model": "efficientnet_b0", "aug": "strong", "finetune_lr": 3e-4},
    {"name": "C_convnext_strong", "model": "convnext_tiny",   "aug": "strong", "finetune_lr": 1e-4},
]

def train(run):
    cmd = [sys.executable, "-m", "foodvision.train",
           "--data-dir", DATA_DIR, "--splits-dir", "data/splits",
           "--exclude-csv", str(DUP_DIR / "exclude.csv"),
           "--output-dir", str(WORK / "runs" / run["name"]),
           "--model", run["model"], "--aug", run["aug"], "--finetune-lr", str(run["finetune_lr"]),
           "--run-name", run["name"], "--num-workers", "2",
           "--tracking-uri", TRACKING_URI, "--mlflow-artifacts", str(WORK / "mlartifacts")]
    if QUICK:
        cmd += ["--head-epochs", "1", "--finetune-epochs", "1", "--max-per-class", "6",
                "--batch-size", "8", "--no-pretrained", "--num-workers", "0"]
    print("\n" + "=" * 70 + f"\n{run['name']}\n" + "=" * 70, flush=True)
    subprocess.run(cmd, check=True)

In [ ]:
train(RUNS[0])

In [ ]:
train(RUNS[1])

In [ ]:
train(RUNS[2])

## 4. Compare runs (MLflow)
The winner is chosen by **validation** top-1 accuracy. Test numbers are shown for reporting, but using them to choose would leak test information into the decision.

In [ ]:
import mlflow
mlflow.set_tracking_uri(TRACKING_URI)
runs = mlflow.search_runs(experiment_names=["food-vision"])
cols = {"tags.mlflow.runName": "run", "params.model": "model", "params.aug": "aug",
        "metrics.val_top1_acc": "val_acc", "metrics.test_top1_acc": "test_acc",
        "metrics.test_top3_acc": "test_top3", "metrics.test_macro_f1": "test_macro_f1",
        "metrics.n_params": "params", "metrics.train_minutes": "minutes"}
summary = (runs[list(cols)].rename(columns=cols)
           .drop_duplicates("run", keep="first").sort_values("val_acc", ascending=False)
           .reset_index(drop=True))
summary["params"] = (summary["params"] / 1e6).round(1).astype(str) + "M"
summary.to_csv(WORK / "experiments_summary.csv", index=False)
summary.round(4)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for run in RUNS:
    h = pd.read_csv(WORK / "runs" / run["name"] / "history.csv")
    axes[0].plot(h.epoch, h.val_acc, "o-", label=run["name"])
    axes[1].plot(h.epoch, h.train_acc - h.val_acc, "o-", label=run["name"])
axes[0].set_title("Validation accuracy"); axes[1].set_title("Overfitting gap (train acc − val acc)")
for ax in axes:
    ax.set_xlabel("Epoch"); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.savefig(WORK / "experiments_comparison.png", dpi=110); plt.show()

## 5. Export the winning model for the API

In [ ]:
best = summary.iloc[0]["run"]
src, dst = WORK / "runs" / best, WORK / "best_artifacts"
shutil.rmtree(dst, ignore_errors=True); dst.mkdir()
for f in ["model.pt", "class_names.json", "model_config.json", "metrics.json", "history.csv",
          "classification_report.txt", "confusion_matrix.png", "confusion_matrix.csv"]:
    shutil.copy(src / f, dst / f)
(dst / "selected_run.json").write_text(json.dumps(
    {"run": best, "selected_by": "val_top1_acc", "git_commit": COMMIT}, indent=2))
print(f"Best run: {best}\n")
print((dst / "classification_report.txt").read_text())

## 6. What to send back / next steps
Download from the **Output** tab: `best_artifacts/`, `experiments_summary.csv`, `experiments_comparison.png`, `duplicates/summary.json`.

**View the MLflow UI on your PC** (optional): download `mlflow.db`, then:
```bash
pip install mlflow
mlflow ui --backend-store-uri sqlite:///mlflow.db
```
Open http://127.0.0.1:5000 to see every parameter and metric curve. Artifact files point to Kaggle paths, so they won't open locally; the metrics and charts will.